# ShieldVoice (SIH26104) — Google Colab GPU Training Pipeline
### AI-Powered Real-Time Detection of Voice Cloning Impersonation Attacks

This notebook provides a complete GPU-accelerated training pipeline:
1. **Datasets**: Downloads ASVspoof 2019, ASVspoof 2021, Deep Voice, and MLAAD via `kagglehub`.
2. **Model**: Fine-tunes **Wav2Vec2-XLS-R (SSL Front-End) + AASIST / Graph Attention Network** for Indian accent robustness and synthetic audio artifact detection.
3. **Persistent Checkpoints**: Auto-saves trained model weights (`best_model.pt`) directly to **Google Drive** (`/content/drive/MyDrive/ShieldVoice_Models`).
4. **CLI Control**: Optional `colab-ssh` tunnel to connect from your local PowerShell/Terminal.

## 1. Verify GPU Acceleration

In [ ]:
!nvidia-smi

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("WARNING: GPU is not enabled! Please go to Runtime -> Change runtime type -> Select T4 GPU or A100.")

## 2. Mount Google Drive for Persistent Storage

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create directory for checkpoint persistence
!mkdir -p /content/drive/MyDrive/ShieldVoice_Models
print("Google Drive mounted successfully! Checkpoints will be saved to: /content/drive/MyDrive/ShieldVoice_Models")

## 3. Clone Repository & Install Dependencies

In [ ]:
# Clone or pull latest codebase
!rm -rf /content/SIH
!git clone https://github.com/kishore-in2007/SIH.git /content/SIH
%cd /content/SIH

# Install dependencies
!pip install -r requirements.txt -q
print("Environment setup complete!")

## 4. Download Kaggle Datasets via `kagglehub`

In [ ]:
import kagglehub

print("Downloading datasets (this may take a few minutes for large archives)...")
asvpoof_2019_path = kagglehub.dataset_download('awsaf49/asvpoof-2019-dataset')
deep_voice_path = kagglehub.dataset_download('birdy654/deep-voice-deepfake-voice-recognition')
avsspoof_2021_path = kagglehub.dataset_download('mohammedabdeldayem/avsspoof-2021')
mlaad_path = kagglehub.dataset_download('trapka/mlaadthe-multi-languagaudioanti-spoofing-dataset')

print("\n--- Data Source Import Complete ---")
print(f"1. ASVspoof 2019: {asvpoof_2019_path}")
print(f"2. Deep Voice:    {deep_voice_path}")
print(f"3. ASVspoof 2021: {avsspoof_2021_path}")
print(f"4. MLAAD:         {mlaad_path}")

## 5. (Optional) Launch SSH CLI Tunnel for Local Terminal Control
Run this cell if you want to SSH into this Colab GPU from your local PowerShell or Terminal.

In [ ]:
from colab_ssh import launch_ssh_cloudflared
# Set your SSH connection password
launch_ssh_cloudflared(password="shieldvoice123")

## 6. Run GPU Training (Wav2Vec2-XLS-R + AASIST Back-End)

In [ ]:
!python train.py \
    --asvspoof19_path "$asvpoof_2019_path" \
    --deepvoice_path "$deep_voice_path" \
    --asvspoof21_path "$avsspoof_2021_path" \
    --mlaad_path "$mlaad_path" \
    --model_type wav2vec2_aasist \
    --ssl_model facebook/wav2vec2-xls-r-300m \
    --epochs 25 \
    --batch_size 16 \
    --lr 1e-4 \
    --ssl_lr 1e-5 \
    --device cuda \
    --save_dir ./saved_models \
    --drive_save_dir /content/drive/MyDrive/ShieldVoice_Models

## 7. Run Benchmark Evaluation (EER, Accuracy, ROC-AUC)

In [ ]:
!python evaluate.py \
    --checkpoint /content/drive/MyDrive/ShieldVoice_Models/best_model.pt \
    --dataset_path "$avsspoof_2021_path" \
    --device cuda

## 8. Test Single Audio Clip Fraud Inference

In [ ]:
import glob

# Locate sample audio clips
audio_samples = glob.glob(f"{deep_voice_path}/**/*.wav", recursive=True) + glob.glob(f"{asvpoof_2019_path}/**/*.flac", recursive=True)
if audio_samples:
    test_clip = audio_samples[0]
    print(f"Testing inference on: {test_clip}")
    !python inference.py --audio "{test_clip}" --checkpoint /content/drive/MyDrive/ShieldVoice_Models/best_model.pt
else:
    print("No sample audio found to test.")

## 9. Plot Training Loss & EER Curves

In [ ]:
import json
import matplotlib.pyplot as plt

history_file = "/content/drive/MyDrive/ShieldVoice_Models/training_history.json"
try:
    with open(history_file, 'r') as f:
        data = json.load(f)
    history = data["history"]

    epochs = [h["epoch"] for h in history]
    train_loss = [h["train_loss"] for h in history]
    val_loss = [h["val_loss"] for h in history]
    val_eer = [h["eer"] for h in history]
    val_acc = [h["accuracy"] for h in history]

    plt.figure(figsize=(14, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_loss, label="Train Loss", marker='o')
    plt.plot(epochs, val_loss, label="Val Loss", marker='s')
    plt.title("Cross-Entropy Loss Curve")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.grid(True)
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, val_eer, label="Equal Error Rate (EER %)", color='red', marker='^')
    plt.plot(epochs, val_acc, label="Validation Accuracy (%)", color='green', marker='x')
    plt.title("Validation Metrics (EER & Accuracy)")
    plt.xlabel("Epoch")
    plt.ylabel("Percentage (%)")
    plt.grid(True)
    plt.legend()

    plt.tight_layout()
    plt.savefig("/content/drive/MyDrive/ShieldVoice_Models/training_metrics_curve.png", dpi=300)
    plt.show()
    print("Loss and EER curves plotted and saved to Google Drive!")
except Exception as e:
    print(f"Could not plot history: {e}")